In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import pandas
data = pandas.read_csv('/content/drive/MyDrive/Health Disease Risk assessment/input/diabetes.csv');

In [ ]:
import pandas as pd
from sklearn import model_selection
import os
os.chdir("/content/drive/MyDrive/Health Disease Risk assessment/src")

import config

print(config)

import os
os.chdir(config.SAVE_TRAINING_FILE_WITH_FOLDS)

if __name__ == "__main__":
    # Training data is in a csv file called train.csv
    df = data

    # we create a new column called kfold and fill it with -1
    df["kfold"] = -1

    # the next step is to randomize the rows of the data
    df = df.sample(frac=1).reset_index(drop=True)

    # fetch targets
    y = df.Outcome.values

    # initiate the kfold class from model_selection module
    kf = model_selection.StratifiedKFold(n_splits=5)

    # fill the new kfold column
    for f, (t_, v_) in enumerate(kf.split(X=df, y=y)):
      df.loc[v_, 'kfold'] = f

    # save the new csv with kfold column
    df.to_csv("diabetes_folds.csv", index=False)

<module 'config' from '/content/drive/MyDrive/Health Disease Risk assessment/src/config.py'>


In [ ]:
data1 = pandas.read_csv("/content/drive/MyDrive/Health Disease Risk assessment/input/diabetes_folds.csv")

In [ ]:
data1.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,kfold
0,2,99,52,15,94,24.6,0.637,21,0,0
1,1,128,88,39,110,36.5,1.057,37,1,0
2,4,132,86,31,0,28.0,0.419,63,0,0
3,6,115,60,39,0,33.7,0.245,40,1,0
4,1,89,24,19,25,27.8,0.559,21,0,0


In [ ]:
data1['kfold'].value_counts()

,count
kfold,
0,154
1,154
2,154
3,153
4,153


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score
import pickle

import os
os.chdir("/content/drive/MyDrive/Health Disease Risk assessment/src")

from config import TRAINING_FILE_WITH_FOLDS, MODEL_OUTPUT
import model_dispacher, param_distributions  # assumes model_dispatcher.py renamed to models.py

from model_dispacher import models
from param_distributions import param_distributions

# Load the dataset with folds
data = pd.read_csv(TRAINING_FILE_WITH_FOLDS)

# Tune model using RandomizedSearchCV
def tune_model(model_name, X, y):
    model = models[model_name]
    if model_name in param_distributions:
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_distributions[model_name],
            n_iter=10,
            cv=5,
            scoring='f1',
            random_state=42,
            n_jobs=-1
        )
        search.fit(X, y)
        return search.best_estimator_, search.best_params_
    else:
        model.fit(X, y)
        return model, None

# Evaluate model on each fold
def evaluate_model_on_folds(model, model_name):
    f1_scores = []
    for fold in range(5):
        df_train = data[data.kfold != fold].reset_index(drop=True)
        df_valid = data[data.kfold == fold].reset_index(drop=True)

        X_train = df_train.drop(["Outcome", "kfold"], axis=1).values
        y_train = df_train.Outcome.values
        X_valid = df_valid.drop(["Outcome", "kfold"], axis=1).values
        y_valid = df_valid.Outcome.values

        model.fit(X_train, y_train)
        preds = model.predict(X_valid)
        f1 = f1_score(y_valid, preds)
        print(f"📁 Fold {fold}: F1 Score = {f1:.4f}")
        f1_scores.append(f1)

    avg_f1 = sum(f1_scores) / len(f1_scores)
    print(f"🎯 Avg F1 Score for {model_name}: {avg_f1:.4f}")
    return avg_f1, f1_scores

if __name__ == "__main__":
    X = data.drop("Outcome", axis=1).values
    y = data.Outcome.values

    best_overall_model = None
    best_overall_score = 0
    best_overall_model_name = ""
    best_overall_params = None

    for model_name in models.keys():
        print(f"\n🔍 Tuning model: {model_name}")
        best_model, best_params = tune_model(model_name, X, y)
        print(f"✅ Best Params: {best_params if best_params else 'Default'}")

        print(f"📊 Evaluating {model_name} on each fold...")
        avg_f1, _ = evaluate_model_on_folds(best_model, model_name)

        if avg_f1 > best_overall_score:
            best_overall_score = avg_f1
            best_overall_model = best_model
            best_overall_model_name = model_name
            best_overall_params = best_params

    # Save the best overall model
    final_model_path = f"{MODEL_OUTPUT}/{best_overall_model_name}_BEST.pkl"
    with open(final_model_path, "wb") as f:
        pickle.dump(best_overall_model, f)

    print(f"\n🏆 Best Overall Model: {best_overall_model_name}")
    print(f"💯 Best F1 Score: {best_overall_score:.4f}")
    print(f"🧪 Best Params: {best_overall_params if best_overall_params else 'Default'}")
    print(f"💾 Model saved to {final_model_path}")



🔍 Tuning model: decision_tree_gini
✅ Best Params: {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 5}
📊 Evaluating decision_tree_gini on each fold...
📁 Fold 0: F1 Score = 0.7222
📁 Fold 1: F1 Score = 0.5918
📁 Fold 2: F1 Score = 0.6078
📁 Fold 3: F1 Score = 0.6087
📁 Fold 4: F1 Score = 0.6061
🎯 Avg F1 Score for decision_tree_gini: 0.6273

🔍 Tuning model: decision_tree_entropy
✅ Best Params: {'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 5}
📊 Evaluating decision_tree_entropy on each fold...
📁 Fold 0: F1 Score = 0.6316
📁 Fold 1: F1 Score = 0.6019
📁 Fold 2: F1 Score = 0.6379
📁 Fold 3: F1 Score = 0.6486
📁 Fold 4: F1 Score = 0.6476
🎯 Avg F1 Score for decision_tree_entropy: 0.6335

🔍 Tuning model: random_forest
✅ Best Params: {'n_estimators': 100, 'min_samples_split': 5, 'max_depth': None}
📊 Evaluating random_forest on each fold...
📁 Fold 0: F1 Score = 0.6517
📁 Fold 1: F1 Score = 0.5941
📁 Fold 2: F1 Score = 0.6327
📁 Fold 3: F1 Score = 0.5794
📁 Fold 4: F1 Score = 0.686

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 9 is smaller than n_iter=10. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


✅ Best Params: {'n_estimators': 100, 'learning_rate': 1}
📊 Evaluating ada_boost on each fold...
📁 Fold 0: F1 Score = 0.5882
📁 Fold 1: F1 Score = 0.6263
📁 Fold 2: F1 Score = 0.6296
📁 Fold 3: F1 Score = 0.5905
📁 Fold 4: F1 Score = 0.7115
🎯 Avg F1 Score for ada_boost: 0.6292

🔍 Tuning model: knn
✅ Best Params: {'weights': 'distance', 'p': 1, 'n_neighbors': 9}
📊 Evaluating knn on each fold...
📁 Fold 0: F1 Score = 0.4819
📁 Fold 1: F1 Score = 0.6000
📁 Fold 2: F1 Score = 0.6019
📁 Fold 3: F1 Score = 0.5872
📁 Fold 4: F1 Score = 0.6600
🎯 Avg F1 Score for knn: 0.5862

🔍 Tuning model: logistic_regression


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
20 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py", line 1203, in fit
    raise ValueError("l1_r

✅ Best Params: {'solver': 'saga', 'penalty': 'l2', 'C': 10}
📊 Evaluating logistic_regression on each fold...
📁 Fold 0: F1 Score = 0.2338
📁 Fold 1: F1 Score = 0.3855
📁 Fold 2: F1 Score = 0.4419
📁 Fold 3: F1 Score = 0.4667
📁 Fold 4: F1 Score = 0.3768
🎯 Avg F1 Score for logistic_regression: 0.3809

🔍 Tuning model: svm_linear
✅ Best Params: {'C': 10}
📊 Evaluating svm_linear on each fold...
📁 Fold 0: F1 Score = 0.5682
📁 Fold 1: F1 Score = 0.5918
📁 Fold 2: F1 Score = 0.5714
📁 Fold 3: F1 Score = 0.6667
📁 Fold 4: F1 Score = 0.7573
🎯 Avg F1 Score for svm_linear: 0.6311

🔍 Tuning model: svm_rbf


...

📁 Fold 1: F1 Score = 0.6186
[LightGBM] [Info] Number of positive: 214, number of negative: 400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000126 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 658
[LightGBM] [Info] Number of data points in the train set: 614, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.348534 -> initscore=-0.625489
[LightGBM] [Info] Start training from score -0.625489
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


📁 Fold 4: F1 Score = 0.6602
🎯 Avg F1 Score for lightgbm: 0.6387

🔍 Tuning model: catboost
0:	learn: 0.6434659	total: 1.46ms	remaining: 145ms
1:	learn: 0.6067543	total: 2.57ms	remaining: 126ms
2:	learn: 0.5798995	total: 3.41ms	remaining: 110ms
3:	learn: 0.5526245	total: 4.21ms	remaining: 101ms
4:	learn: 0.5321820	total: 5ms	remaining: 94.9ms
5:	learn: 0.5168417	total: 5.86ms	remaining: 91.9ms
6:	learn: 0.5020907	total: 6.61ms	remaining: 87.8ms
7:	learn: 0.4920803	total: 7.4ms	remaining: 85.1ms
8:	learn: 0.4817336	total: 8.21ms	remaining: 83ms
9:	learn: 0.4746973	total: 9.03ms	remaining: 81.3ms
10:	learn: 0.4677496	total: 9.86ms	remaining: 79.8ms
11:	learn: 0.4612871	total: 10.7ms	remaining: 78.1ms
12:	learn: 0.4554428	total: 11.5ms	remaining: 77ms
13:	learn: 0.4511102	total: 12.3ms	remaining: 75.3ms
14:	learn: 0.4462688	total: 13.1ms	remaining: 74.4ms
15:	learn: 0.4381853	total: 13.9ms	remaining: 72.8ms
16:	learn: 0.4344813	total: 14.7ms	remaining: 71.7ms
17:	learn: 0.4305062	total: 15.

In [ ]:
import pickle

with open('/content/drive/MyDrive/Health Disease Risk assessment/Models/catboost_BEST.pkl', 'rb') as file:
    model = pickle.load(file)

In [ ]:
data

array([[  1.   , 180.   ,   0.   , ...,   0.282,  41.   ,   1.   ],
       [  2.   , 121.   ,  70.   , ...,   0.886,  23.   ,   0.   ],
       [  0.   , 129.   ,  80.   , ...,   0.703,  29.   ,   0.   ],
       ...,
       [  0.   ,  57.   ,  60.   , ...,   0.735,  67.   ,   0.   ],
       [  2.   ,  89.   ,  90.   , ...,   0.292,  42.   ,   0.   ],
       [  8.   , 197.   ,  74.   , ...,   1.191,  39.   ,   1.   ]])

In [ ]:
data = data.values

In [ ]:
data = data.drop("Outcome", axis=1).values

In [ ]:
data

array([[  6.   , 148.   ,  72.   , ...,  33.6  ,   0.627,  50.   ],
       [  1.   ,  85.   ,  66.   , ...,  26.6  ,   0.351,  31.   ],
       [  8.   , 183.   ,  64.   , ...,  23.3  ,   0.672,  32.   ],
       ...,
       [  5.   , 121.   ,  72.   , ...,  26.2  ,   0.245,  30.   ],
       [  1.   , 126.   ,  60.   , ...,  30.1  ,   0.349,  47.   ],
       [  1.   ,  93.   ,  70.   , ...,  30.4  ,   0.315,  23.   ]])

In [ ]:
data[1]

array([ 1.   , 85.   , 66.   , 29.   ,  0.   , 26.6  ,  0.351, 31.   ,
        0.   ])

In [ ]:
# prompt: convert a list into numpy array

import numpy as np

# Assuming 'data' is your pandas DataFrame and you want to convert a specific column to a NumPy array
# Example: Convert the first row of 'data' to a NumPy array.
numpy_array = np.array(data[1])

print(numpy_array)
print(type(numpy_array))


In [ ]:
data

array([[  6.   , 148.   ,  72.   , ...,   0.627,  50.   ,   1.   ],
       [  1.   ,  85.   ,  66.   , ...,   0.351,  31.   ,   0.   ],
       [  8.   , 183.   ,  64.   , ...,   0.672,  32.   ,   1.   ],
       ...,
       [  5.   , 121.   ,  72.   , ...,   0.245,  30.   ,   0.   ],
       [  1.   , 126.   ,  60.   , ...,   0.349,  47.   ,   1.   ],
       [  1.   ,  93.   ,  70.   , ...,   0.315,  23.   ,   0.   ]])

In [ ]:
data

array(None, dtype=object)

In [ ]:
import numpy as np
model.predict(np.array(data))

array([1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0,
       1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0,
       1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1,